# MiniGPT, grown: follow along

This workbook goes with my post [MiniGPT (Part 2)](https://haddley.github.io/posts/minigpt-grown/). It grows my MiniGPT *exhibit* model from random numbers, using the guessing game, and reproduces the post's numbers on the way: the surprise score, the ladder of four ways to shape the wheel, and the exhibit growing, step by step. At the end, it compares what it grew with my published exhibit, number by number.

Run the cells from top to bottom on a CPU runtime. Everything is quick except the growing itself, which is 3,000 training steps: about 6 minutes on my Mac Studio's CPU, and longer on a free Colab CPU.

The model code in section 1 is copied from Jibin Joseph's [MiniGPT notebook](https://github.com/jibin10/MiniGPT), sections 1.1 to 1.5, under its MIT licence (notice at the end). The companion workbook for running the finished model is [minigpt_follow_along.ipynb](https://colab.research.google.com/github/Haddley/minigpt-series/blob/main/part1-running/minigpt_follow_along.ipynb).

## 1. The model code

The notebook's own classes, unchanged. They define the machine; training fills in its numbers.

In [1]:
import math, time, urllib.request
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
@dataclass
class GPTConfig:
    # Maximum number of tokens the model can look at at once.
    block_size: int = 128
    # Number of tokens in the vocabulary - for Part 1 testing.
    vocab_size: int = 65
    # Number of Transformer blocks stacked in the model.
    n_layer: int = 4
    # Number of attention heads inside each self-attention module.
    n_head: int = 4
    # Size of each token embedding vector.
    n_embd: int = 128
    # Dropout probability used for regularization.
    dropout: float = 0.1

In [3]:
# Causal multi-head self-attention module.
class CausalSelfAttention(nn.Module):
    # Initialize the attention module.
    def __init__(self, config):
        super().__init__()
        # Check that the embedding dimension can be evenly split across attention heads.
        assert config.n_embd % config.n_head == 0, "n_embd must be divisible by n_head"

        # Store the number of attention heads.
        self.n_head = config.n_head
        # Store the embedding dimension.
        self.n_embd = config.n_embd
        # Compute the size of each attention head.
        self.head_dim = config.n_embd // config.n_head

        # Create the query, key, value projection layers.
        self.query = nn.Linear(config.n_embd, config.n_embd)
        self.key = nn.Linear(config.n_embd, config.n_embd)
        self.value = nn.Linear(config.n_embd, config.n_embd)

        # Create the output projection layer after attention heads are combined.
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        # Create dropout for attention probabilities and after the output projection.
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # Create a lower-triangular causal mask of shape (block_size, block_size).
        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size, dtype=torch.bool))
        # Reshape the mask to broadcast over batch and heads: (1, 1, block_size, block_size).
        causal_mask = causal_mask.view(1, 1, config.block_size, config.block_size)

        # Register the mask as a buffer so it moves with the model but is not a trainable parameter.
        self.register_buffer("causal_mask", causal_mask)

    # Define the forward pass.
    def forward(self, x):
        # Read batch size B, sequence length T, and embedding size C from input.
        B, T, C = x.shape
        # Project input embeddings into query, key, value vectors.
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Reshape queries, keys, values from (B, T, C) to (B, T, n_head, head_dim).
        q = q.view(B, T, self.n_head, self.head_dim)
        k = k.view(B, T, self.n_head, self.head_dim)
        v = v.view(B, T, self.n_head, self.head_dim)

        # Move the head dimension before the time dimension: (B, n_head, T, head_dim).
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Compute raw attention scores using query-key dot products.
        scores = q @ k.transpose(-2, -1)

        # Scale attention scores by sqrt(head_dim) for stable softmax values.
        scores = scores / math.sqrt(self.head_dim)
        # Select only the part of the causal mask needed for the current sequence length T.
        mask = self.causal_mask[:, :, :T, :T]

        # Put -infinity where the model should not attend, especially future positions.
        scores = scores.masked_fill(mask == False, float("-inf"))

        # Convert attention scores into attention probabilities.
        attn = F.softmax(scores, dim=-1)
        # Apply dropout to attention probabilities.
        attn = self.attn_dropout(attn)
        # Multiply attention probabilities by value vectors to get context vectors.
        out = attn @ v

        # Move the time dimension back before the head dimension: (B, T, n_head, head_dim).
        out = out.transpose(1, 2)
        # Make tensor memory layout contiguous before reshaping.
        out = out.contiguous()
        # Merge all attention heads back into the embedding dimension: (B, T, C).
        out = out.view(B, T, C)
        # Apply final linear projection after combining heads.
        out = self.proj(out)
        # Apply residual dropout after projection.
        out = self.resid_dropout(out)

        # Return the attention output.
        return out

In [4]:
# Feed-forward MLP used inside each Transformer block.
class FeedForward(nn.Module):
    # Initialize the MLP.
    def __init__(self, config):
        super().__init__()

        # First linear layer expands the embedding dimension by 4x.
        self.fc1 = nn.Linear(config.n_embd, 4 * config.n_embd)
        # GELU activation is commonly used in GPT-style Transformers.
        self.gelu = nn.GELU()
        # Second linear layer projects the hidden dimension back to n_embd.
        self.fc2 = nn.Linear(4 * config.n_embd, config.n_embd)
        # Dropout is used for regularization.
        self.dropout = nn.Dropout(config.dropout)

    # Define the forward pass for the MLP.
    def forward(self, x):
        # Apply the first linear layer.
        x = self.fc1(x)
        # GELU nonlinearity.
        x = self.gelu(x)
        # Second linear layer.
        x = self.fc2(x)
        # Dropout.
        x = self.dropout(x)

        # Return the MLP output.
        return x

In [5]:
# Decoder-only Transformer block.
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        # LayerNorm is applied before self-attention.
        self.ln1 = nn.LayerNorm(config.n_embd)

        # Causal self-attention module.
        self.attn = CausalSelfAttention(config)

        # Second LayerNorm is applied before the MLP.
        self.ln2 = nn.LayerNorm(config.n_embd)

        # Feed-forward MLP module.
        self.mlp = FeedForward(config)

    # Forward pass for one Transformer block.
    def forward(self, x):
        # LayerNorm, then attention, then add the residual connection.
        x = x + self.attn(self.ln1(x))

        # LayerNorm, then MLP, then add the residual connection.
        x = x + self.mlp(self.ln2(x))

        # Updated hidden states.
        return x

In [6]:
class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        # Token embedding table maps token IDs to vectors.
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)

        # Positional embedding table maps token positions to vectors.
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)

        # Dropout is applied after adding token and position embeddings.
        self.dropout = nn.Dropout(config.dropout)

        # List of Transformer blocks.
        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layer)
        ])

        # Final LayerNorm before the language-model head.
        self.final_ln = nn.LayerNorm(config.n_embd)

        # Linear language-model head maps hidden states to vocabulary logits.
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)

        # Custom weight initialization to all suitable layers.
        self.apply(self._init_weights)

    # Weight initialization function.
    def _init_weights(self, module):
        # Check whether the module is a Linear layer.
        if isinstance(module, nn.Linear):
            # Initialize Linear layer weights from a normal distribution.
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            # Check whether the Linear layer has a bias term.
            if module.bias is not None:
                # Initialize Linear layer bias values to zero.
                nn.init.zeros_(module.bias)

        # Check whether the module is an Embedding layer.
        elif isinstance(module, nn.Embedding):
            # Initialize Embedding weights from a normal distribution.
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # Count the total number of trainable model parameters.
    def count_parameters(self):
        # Sum the number of elements in every trainable parameter tensor.
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # Forward pass of the GPT model.
    def forward(self, idx, targets=None):
        # Read batch size B and sequence length T from token ID tensor.
        B, T = idx.shape

        # Make sure the input sequence is not longer than the model context length.
        assert T <= self.config.block_size, "Input sequence is longer than block_size"

        # Create position IDs from 0 to T-1 on the same device as the input.
        pos = torch.arange(0, T, device=idx.device)

        # Convert token IDs into token embeddings of shape (B, T, n_embd).
        tok_emb = self.token_embedding(idx)

        # Convert position IDs into position embeddings of shape (T, n_embd).
        pos_emb = self.position_embedding(pos)

        # Add token embeddings and position embeddings using broadcasting.
        x = tok_emb + pos_emb

        # Dropout after the embedding sum.
        x = self.dropout(x)

        # Pass the hidden states through each Transformer block.
        for block in self.blocks:
            # Update hidden states using one Transformer block.
            x = block(x)

        # Final LayerNorm.
        x = self.final_ln(x)

        # Convert final hidden states into vocabulary logits.
        logits = self.lm_head(x)

        # Start with no loss by default.
        loss = None

        # Compute loss only when target tokens are provided.
        if targets is not None:
            # Flatten logits from (B, T, vocab_size) to (B*T, vocab_size).
            logits_flat = logits.view(B * T, self.config.vocab_size)

            # Flatten targets from (B, T) to (B*T).
            targets_flat = targets.view(B * T)

            # Compute cross-entropy loss for next-token prediction.
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

## 2. The practice text

Tiny Shakespeare, about 1.1 million letters, from the same address the notebook uses. The vocabulary is every letter in it, sorted, and the last 10% is locked away as the exam text, which training never sees.

In [7]:
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "input.txt")
text = open("input.txt", encoding="utf-8").read()

chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
def decode(ids): return "".join(chars[i] for i in ids)

data = torch.tensor([stoi[ch] for ch in text])
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"{len(text):,} letters, {V} different letters")
print(f"practice text: {len(train_data):,} letters; locked-away exam text: {len(val_data):,} letters")

1,115,394 letters, 65 different letters
practice text: 1,003,854 letters; locked-away exam text: 111,540 letters


## 3. Starting from nothing

A freshly built machine has random numbers on every dial, so its wheel is almost even: every letter gets a chance close to 1 in 65.

In [8]:
torch.manual_seed(42)
untrained = MiniGPT(GPTConfig(vocab_size=V))
untrained.eval()
with torch.no_grad():
    logits, _ = untrained(torch.tensor([[stoi[ch] for ch in "goo"]]))
p = torch.softmax(logits[0, -1], dim=-1)
print(f"chances after 'goo': between {p.min():.1%} and {p.max():.1%}; an even wheel would give {1/V:.1%} each")

torch.manual_seed(0)
idx = torch.tensor([[stoi[ch] for ch in "ROMEO:"]])
with torch.no_grad():
    for _ in range(80):
        logits, _ = untrained(idx)
        idx = torch.cat([idx, torch.multinomial(torch.softmax(logits[0, -1], -1), 1).view(1, 1)], dim=1)
print(decode(idx[0].tolist()))

chances after 'goo': between 0.9% and 2.8%; an even wheel would give 1.5% each
ROMEO:t3ZxjmhPkqkARxLsXltUWRoHcu ?eThjyU&aNdkIgAlsvSBUMycoRu$BygX- ntw
kVrM h&?j DJRbs


## 4. The surprise score

The surprise score is ln(1 ÷ chance): 0 for a chance of 100%, and 0.69 more for every halving. To look at real chances, load my trained exhibit model.

In [9]:
for halvings in range(7):
    chance = 0.5 ** halvings
    print(f"chance {chance:7.2%}   surprise score {math.log(1 / chance):.2f}")
print(f"\nan even wheel, 1 in 65: surprise score {math.log(V):.2f}")

chance 100.00%   surprise score 0.00
chance  50.00%   surprise score 0.69
chance  25.00%   surprise score 1.39
chance  12.50%   surprise score 2.08
chance   6.25%   surprise score 2.77
chance   3.12%   surprise score 3.47
chance   1.56%   surprise score 4.16

an even wheel, 1 in 65: surprise score 4.17


In [10]:
urllib.request.urlretrieve("https://haddley.github.io/minigpt-demo/exhibit.pt", "exhibit.pt")
exhibit = MiniGPT(GPTConfig(vocab_size=V))
exhibit.load_state_dict(torch.load("exhibit.pt", map_location="cpu"))
exhibit.eval()
exhibit.requires_grad_(False)

logits, _ = exhibit(torch.tensor([[stoi[ch] for ch in "good m"]]))
p = torch.softmax(logits[0, -1], dim=-1)
for ch in "yea":
    print(f"after 'good m', {ch!r} gets {p[stoi[ch]]:.1%}: surprise score {math.log(1 / p[stoi[ch]].item()):.2f}")

after 'good m', 'y' gets 40.8%: surprise score 0.90
after 'good m', 'e' gets 26.9%: surprise score 1.31
after 'good m', 'a' gets 15.2%: surprise score 1.88


## 5. Climbing the ladder

Four ways to shape the wheel, each scored on the locked-away exam text. Rungs 1 to 3 need no training at all, only counting. One random-number generator draws all three samples, in this order, so they match the post.

In [11]:
g = torch.Generator().manual_seed(7)

# rung 1: an even wheel
print("rung 1, even wheel:   ", round(math.log(V), 2))
print(decode(torch.randint(0, V, (160,), generator=g).tolist())[:56], "\n")

# rung 2: how common each letter is
counts = torch.bincount(train_data, minlength=V).float() + 1
p1 = counts / counts.sum()
print("rung 2, letter counts:", round((-torch.log(p1[val_data])).mean().item(), 2))
print("biggest slices:", [(chars[i], f"{p1[i]:.1%}") for i in p1.topk(4).indices.tolist()])
print(decode(torch.multinomial(p1, 160, replacement=True, generator=g).tolist())[:80], "\n")

# rung 3: count pairs, so the wheel depends on the letter before
pairs = torch.ones(V, V)
pairs.index_put_((train_data[:-1], train_data[1:]), torch.ones(len(train_data) - 1), accumulate=True)
P = pairs / pairs.sum(1, keepdim=True)
print("rung 3, pair counts:  ", round((-torch.log(P[val_data[:-1], val_data[1:]])).mean().item(), 2))
q = stoi["q"]
print("after q:", int(pairs[q].sum().item() - V), "times in the practice text,",
      int(pairs[q, stoi["u"]].item() - 1), "of them followed by u")
s = [stoi["\n"]]
for _ in range(240):
    s.append(torch.multinomial(P[s[-1]], 1, generator=g).item())
print(decode(s[1:])[:160])

rung 1, even wheel:    4.17
g?rr$s!fywKSFdxH;Yty,qM,F$ZOg.r:DFM DnlCk,QT!N&Uss3pJtv- 

rung 2, letter counts: 3.35
biggest slices: [(' ', '15.3%'), ('e', '8.5%'), ('t', '6.0%'), ('o', '5.9%')]
uooy
sbp ;cTIcdbota rMnufKthlnfmsnnretuUta 
 vs  nae
hEyIlseraoignriS elsiik
eor 

rung 3, pair counts:   2.48
after q: 563 times in the practice text, 563 of them followed by u

SH: lwh, s ts at a ary okm by myore an suche ns hur,
Ancayontouna ondevin: t prt igr abee moo dase use r chm boutome son thef anshye g orinomod
O:
DWhimilito m


Is training just a fancy way of counting? Give a machine one dial for every pair of letters, all set to 0, and train it with the guessing game:

In [12]:
torch.manual_seed(0)
pair_dials = nn.Embedding(V, V)
nn.init.zeros_(pair_dials.weight)
opt = torch.optim.AdamW(pair_dials.parameters(), lr=0.1, weight_decay=0)
t = time.time()
for step in range(3000):
    ix = torch.randint(0, len(train_data) - 1, (4096,))
    loss = F.cross_entropy(pair_dials(train_data[ix]), train_data[ix + 1])
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    learned = F.cross_entropy(pair_dials(val_data[:-1]), val_data[1:]).item()
    after_q = torch.softmax(pair_dials.weight[q], -1)
print(f"trained in {time.time() - t:.1f} seconds; surprise score {learned:.2f}; after q, u gets {after_q[stoi['u']]:.0%}")

trained in 2.6 seconds; surprise score 2.49; after q, u gets 100%


Rung 4: my exhibit MiniGPT, which can look up to 128 letters back, scored the same way on the exam text:

In [13]:
def exam_score(model, stride=64):   # every 64th snippet of the exam text
    scores = []
    with torch.no_grad():
        for k in range(0, len(val_data) - 129, 128 * stride):
            ix = torch.arange(k, min(k + 128 * stride, len(val_data) - 129), 128)
            x = torch.stack([val_data[i:i + 128] for i in ix])
            y = torch.stack([val_data[i + 1:i + 129] for i in ix])
            scores.append(model(x, y)[1].item())
    return sum(scores) / len(scores)

s4 = exam_score(exhibit)
print(f"rung 4, my exhibit:    {s4:.2f}, as unsure as choosing between {math.exp(s4):.1f} letters")

rung 4, my exhibit:    1.70, as unsure as choosing between 5.5 letters


## 6. Growing the exhibit

This is exactly how I grew the exhibit: the notebook's model and settings, 3,000 steps of 32 snippets of 128 letters, with every random choice fixed in advance. Five times along the way, it stops to sit the exam, write from `ROMEO:`, and report the start of the `g` card and the chance of `d` after `goo`.

In [14]:
torch.use_deterministic_algorithms(True)
torch.set_num_threads(4)

torch.manual_seed(42)
model = MiniGPT(GPTConfig(block_size=128, vocab_size=V, n_layer=4, n_head=4, n_embd=128, dropout=0.1))
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
batch_rng = torch.Generator().manual_seed(1337)

def snapshot(step):
    model.eval()
    rng = torch.random.get_rng_state()          # so that snapshots do not change the training run
    with torch.no_grad():
        score = exam_score(model)
        sample_rng = torch.Generator().manual_seed(5)
        idx = torch.tensor([[stoi[ch] for ch in "ROMEO:"]])
        for _ in range(150):
            logits, _ = model(idx[:, -128:])
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits[:, -1, :] / 0.8, -1), 1, generator=sample_rng)], 1)
        logits, _ = model(torch.tensor([[stoi[ch] for ch in "goo"]]))
        d = F.softmax(logits[0, -1], -1)[stoi["d"]].item()
    torch.random.set_rng_state(rng)
    model.train()
    g_card = ", ".join(f"{v:.3f}" for v in model.token_embedding.weight[stoi["g"], :4].tolist())
    print(f"--- step {step}: exam score {score:.2f}; g card begins {g_card}; d after goo {d:.1%}")
    print(decode(idx[0].tolist()), "\n")

t = time.time()
for step in range(3001):
    if step in (0, 100, 300, 1000, 3000):
        snapshot(step)
    if step == 3000:
        break
    ix = torch.randint(len(train_data) - 128, (32,), generator=batch_rng)
    x = torch.stack([train_data[i:i + 128] for i in ix])
    y = torch.stack([train_data[i + 1:i + 129] for i in ix])
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"grown in {(time.time() - t) / 60:.1f} minutes")

--- step 0: exam score 4.17; g card begins -0.048, 0.010, -0.012, -0.008; d after goo 2.0%
ROMEO:hAbB
n, !ne3XobBELz IxIy,ZS.mopvcouAGng
$BDdCxXCzStrk-Mt!FDGq3HxzntxYiR;Y!wTgFoe&b!!.yn!ehKhjW$,Q
!!ctrjRUVTcaFoMg.pOw:GCRI$wZ!oAWm3JcwW,a!AWPpeJl:tm, 



--- step 100: exam score 2.64; g card begins -0.044, 0.013, -0.002, 0.008; d after goo 3.8%
ROMEO:
Abk w, thel orle s I I lly momenourang
Whe te t stheamt thes wianthouREYalTgeoe bee y ee Khad de
!
I wofomenatomy.
O:
Gith wy oulmamawheave the letho 



--- step 300: exam score 2.45; g card begins -0.043, 0.015, 0.000, 0.010; d after goo 3.4%
ROMEO:
And no thearors t withe, y momerour ng
Whe te thotrkimt the ar anthou chat geee bee y ee inod de
Angtheromenatong.
O:
Goth wn o thad wheave the llt s 



--- step 1000: exam score 2.07; g card begins -0.034, 0.005, -0.021, 0.010; d after goo 39.5%
ROMEO:
Ay thy the mort to she fromme may tong
Whe this strord.

DUCENIUS:
Ye chat gooe be ty een had de
And with micting.
OLOLAR:
will you car, you per, tho 



--- step 3000: exam score 1.70; g card begins -0.049, 0.032, 0.014, 0.030; d after goo 96.6%
ROMEO:
Ay, what the blood stisficemed:
And not we to their all the eather of day age,
As why earthy see
Against becaused. O now thun of my care to the let o 

grown in 6.2 minutes


## 7. The payoff: is it the exhibit?

Compare every one of the 826,433 grown numbers with my published exhibit. On my Mac Studio's CPU they match exactly. A different computer does some of its arithmetic in a slightly different order, so on Colab expect a machine that is very close, but not identical to the last digit.

In [15]:
grown = model.state_dict()
mine = torch.load("exhibit.pt", map_location="cpu")
same = sum(torch.equal(grown[k], mine[k]) for k in mine)
biggest = max((grown[k].float() - mine[k].float()).abs().max().item() for k in mine)
print(f"{same} of {len(mine)} named sets of numbers identical; biggest difference {biggest:.2g}")

model.eval()
with torch.no_grad():
    logits, _ = model(torch.tensor([[stoi[ch] for ch in "goo"]]))
print(f"grown model: d after goo {torch.softmax(logits[0, -1], -1)[stoi['d']]:.1%}; "
      f"exam score {exam_score(model):.2f}")

74 of 74 named sets of numbers identical; biggest difference 0


grown model: d after goo 96.6%; exam score 1.70


---
### Licence notice for the model code in section 1

MiniGPT, Copyright (c) 2026 Jibin Joseph, https://github.com/jibin10/MiniGPT. The code is licensed under the MIT License:

Permission is hereby granted, free of charge, to any person obtaining a copy of the software and associated documentation files covered by the MIT License (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.